In [1]:
import pandas as pd
# from LabData.DataLoaders.GutMBLoader import GutMBLoader
# from LabData.DataLoaders.SubjectLoader import SubjectLoader
# from LabData.DataLoaders.DietLoggingLoader import DietLoggingLoader
# from LabData.DataAnalyses.TenK_Trajectories.utils import get_diet_logging_around_stage
import seaborn as sns
import numpy as np
from scipy import stats
import matplotlib.pyplot as plt
import pickle
import lightgbm as lgb
import math
import scipy.stats as stats
from statsmodels.stats.multitest import multipletests
from scipy.stats import spearmanr
import re
from LabData.DataLoaders.BodyMeasuresLoader import BodyMeasuresLoader
from LabData.DataLoaders.LifeStyleLoader import LifeStyleLoader
from LabData.DataLoaders.DemographicsLoader import DemographicsLoader
from LabData.DataLoaders.Medications10KLoader import Medications10KLoader



In [2]:
home_path = '/net/mraid20/export/genie/LabData/Analyses/tomerse/diet_mb/'
SPECIES = 'segal_species' # 'segal_species' or 'mpa_species'
color1 = "#66C2A5"
single_style = "nature_single.mplstyle"
double_style = "nature_double.mplstyle"
third_style = "nature_third.mplstyle"
plt.rcParams["figure.dpi"] = 150
plt.style.use(single_style)
study_ids=[10, 1001, 1002, 1003, 1004, 1005, 1006, 1007, 1008, 1009, 1010]
stage = 'baseline'

In [10]:
def read_results(df):
    output = []
    for col in df.columns:
        output.append(df[col])
    return tuple(output)

In [3]:
species = '' if SPECIES == 'segal_species' else '_mpa'

diet_mb = pd.read_pickle(home_path + f"data/{SPECIES}/diet_mb.pkl")
with open(home_path + f'data/{SPECIES}/my_lists.pkl', 'rb') as file:
    loaded_lists = pickle.load(file)
base_features, all_features, targets = loaded_lists
with open(home_path + f'data/{SPECIES}/scaler.pkl', 'rb') as scaler_file:
        scaler = pickle.load(scaler_file)
diet_mb

,Acorn squash,Alfalfa sprouts,Almond Beverage,Almond flour,Almond spread,Almonds,Amba,Apple,Apple Cake,Apple Vinegar,...,Irritable Bowel Syndrome (IBS),Peptic Ulcer Disease,Gallstone disease,Asthma,Atopic dermatitis,Psoriasis,Allergy,Depression,Anxiety,Hypothyroidism
RegistrationCode,,,,,,,,,,,,,,,,,,,,,
10K_1000942861,0.0,0.0,0.0,0.0,0.0,0.000000,0.0,0.016416,0.000000,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,1.0,0.0,0.0
10K_1001201093,0.0,0.0,0.0,0.0,0.0,0.000000,0.0,0.013895,0.000000,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
10K_1002254441,0.0,0.0,0.0,0.0,0.0,0.081238,0.0,0.000000,0.000000,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
10K_1003113258,0.0,0.0,0.0,0.0,0.0,0.002048,0.0,0.008929,0.000000,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
10K_1007330152,0.0,0.0,0.0,0.0,0.0,0.000000,0.0,0.000000,0.000000,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
10K_9996884777,0.0,0.0,0.0,0.0,0.0,0.002401,0.0,0.000000,0.000000,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
10K_9998420917,0.0,0.0,0.0,0.0,0.0,0.019995,0.0,0.018783,0.000000,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
10K_9998635752,0.0,0.0,0.0,0.0,0.0,0.001607,0.0,0.011305,0.000000,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0


### BMI

In [12]:
bml = BodyMeasuresLoader()
# bmld = bml.get_data(study_ids=study_ids, cols=['weight', 'height', 'bmr'])
bmld = bml.get_data(study_ids=study_ids)
bmldf = bmld.df
bmldf.info()

<class 'pandas.core.frame.DataFrame'>
MultiIndex: 25257 entries, ('10K_1000942861', Timestamp('2021-12-13 07:09:38.954985')) to ('10K_9999409119', Timestamp('2020-06-09 06:35:10.133884'))
Data columns (total 35 columns):
 #   Column                                        Non-Null Count  Dtype  
---  ------                                        --------------  -----  
 0   abdominal                                     0 non-null      float64
 1   standing_three_min_blood_pressure_systolic    15857 non-null  float64
 2   weight                                        25176 non-null  float64
 3   hips                                          25195 non-null  float64
 4   is_getting_period                             1611 non-null   object 
 5   lying_blood_pressure_diastolic                15928 non-null  float64
 6   dizziness                                     15929 non-null  object 
 7   waist                                         25196 non-null  float64
 8   lying_blood_pressure_pul

In [13]:
if stage == 'baseline':
    # Keep only baseline
    bmldf = bmldf[~bmldf.index.get_level_values(0).duplicated()]
    bmldf = bmldf.reset_index(level=[1], drop=True)
elif stage == '02_00_visit':
    # Keep only the second entry (2nd visit)
    bmldf = bmldf.groupby(level=0).nth(1) 
elif stage == '04_00_visit':
    # Keep only the second entry (3rd visit)
    bmldf = bmldf.groupby(level=0).nth(1) 
bmldf

,abdominal,standing_three_min_blood_pressure_systolic,weight,hips,is_getting_period,lying_blood_pressure_diastolic,dizziness,waist,lying_blood_pressure_pulse_rate,number_of_days_in_cycle,...,standing_three_min_blood_pressure_pulse_rate,sitting_blood_pressure_systolic,body_fat,whr,frequency_of_period,standing_three_min_blood_pressure_diastolic,standing_one_min_blood_pressure_pulse_rate,sitting_blood_pressure_diastolic,height,bmr
RegistrationCode,,,,,,,,,,,,,,,,,,,,,
10K_1000942861,NaN,130.0,91.800003,107.000000,NaN,87.0,False,99.0,50.0,NaN,...,66.0,NaN,NaN,0.925234,NaN,90.0,69.0,85.0,180.500000,NaN
10K_1001201093,NaN,102.0,59.400002,97.000000,NaN,69.0,False,76.0,66.0,NaN,...,77.0,104.0,NaN,0.783505,NaN,74.0,80.0,72.0,170.000000,NaN
10K_1002033709,NaN,105.0,55.000000,91.000000,NaN,72.0,True,80.0,60.0,NaN,...,78.0,102.0,NaN,0.879121,NaN,78.0,82.0,71.0,159.000000,NaN
10K_1002087123,NaN,124.0,105.699997,119.000000,Yes,89.0,False,113.0,72.0,27.0,...,87.0,145.0,45.099998,0.949580,NaN,92.0,81.0,104.0,169.800003,1805.0
10K_1002254441,NaN,121.0,74.000000,95.000000,NaN,69.0,False,90.0,66.0,NaN,...,80.0,107.0,NaN,0.947368,NaN,80.0,77.0,66.0,178.000000,NaN
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
10K_9998418497,NaN,83.0,66.000000,100.000000,Yes,61.0,True,87.0,71.0,28.0,...,87.0,93.0,32.200001,0.870000,NaN,66.0,87.0,62.0,168.300003,1343.0
10K_9998420917,NaN,NaN,56.200001,93.000000,NaN,NaN,NaN,67.0,NaN,NaN,...,NaN,97.0,NaN,0.720430,NaN,NaN,NaN,55.0,162.000000,NaN
10K_9998635752,NaN,NaN,67.000000,94.199997,NaN,NaN,NaN,81.0,NaN,NaN,...,NaN,118.0,NaN,0.859873,NaN,NaN,NaN,64.0,173.000000,NaN


In [14]:
diet_mb = diet_mb.join(bmldf['bmi'], how='inner')

### Comorbidities

In [15]:
baseline_conditions = pd.read_csv("/net/mraid20/export/genie/LabData/Data/10K/for_review/baseline_conditions_all.csv")
follow_up_conditions = pd.read_csv("/net/mraid20/export/genie/LabData/Data/10K/for_review/follow_up_conditions_all.csv")
baseline_conditions

,RegistrationCode,medical_condition,Date,Start,name,created_at,research_stage,Group,Consolidated name,english_name,index,Baseline_Date
0,10K_1000942861,BlockL2-6A7,2008-01-01,True,דיכאון,2021-06-11 11:51:55,baseline,Neurologic,Depression,Depressive disorders,NaN,NaN
1,10K_1000942861,DB60,2000-01-01,True,טחורים,2021-06-11 11:51:55,baseline,Surgery,Haemorrhoids,Haemorrhoids,NaN,NaN
2,10K_1002033709,ED80,2000-01-01,True,אקנה,2022-01-11 07:16:11.594000,baseline,Dermatology,Allergy,Acne,NaN,NaN
3,10K_1002033709,GC08.Z,2000-01-01,True,דלקת בשתן,2022-01-11 07:16:11.594000,baseline,Urology,Urinary tract infection,"Urinary tract infection, site and agent not sp...",NaN,NaN
4,10K_1002087123,6A05,2000-01-01,True,הפרעות קשב וריכוז,2019-10-25 12:55:22,baseline,Neurologic,Attention Deficit Disorder (ADHD),Attention deficit hyperactivity disorder,NaN,NaN
...,...,...,...,...,...,...,...,...,...,...,...,...
39649,10K_9985863396,BA00,2015-01-01 00:00:00.000000,True,יתר לחץ דם,2021-08-15 13:59:03.554201+00:00,01_00_call,Cardiovascular,Hypertension,Essential hypertension,1637.0,2020-01-13 07:09:54.773723
39650,10K_9985863396,CA22,2015-01-01 00:00:00.000000,True,copd,2021-08-15 13:59:03.554312+00:00,01_00_call,Pulmonology,COPD,Chronic obstructive pulmonary disease,1638.0,2020-01-13 07:09:54.773723
39651,10K_9985863396,DA22,2015-01-01 00:00:00.000000,True,צרבת,2021-08-15 13:59:03.554534+00:00,01_00_call,Gastro,Peptic Ulcer Disease,Gastro-oesophageal reflux disease,1640.0,2020-01-13 07:09:54.773723
39652,10K_9988216757,RA01,2021-08-17 00:00:00.000000,True,קורונה,2023-08-10 17:05:27.256570+00:00,01_00_call,Infectious Disease,COVID-19,COVID-19,17379.0,2022-07-31 10:10:29.345662


In [16]:
baseline_conditions["Group"].value_counts()

Metabolic             5406
Neurologic            4304
Orthopedic            3642
Gastro                3534
Surgery               3408
Dermatology           1977
Cardiovascular        1920
Immunology            1855
Urology               1716
Endocrinology         1714
ENT                   1249
Infectious Disease    1022
Hematological          992
Pulmonology            715
OBGyn                  669
Rheumatology           613
Sleep                  548
Oncology               510
Other                  466
Eye disorder           337
Neurologic               8
Neprology                5
Name: Group, dtype: int64

In [17]:
baseline_conditions["Consolidated name"].value_counts().head(30)

Allergy                              2936
Hyperlipidemia                       2652
Haemorrhoids                         2463
Back pain                            2384
Attention Deficit Disorder (ADHD)    2266
Peptic Ulcer Disease                 1377
Hypertension                         1351
Urinary tract infection              1255
Fracture                             1098
Prediabetes                          1015
COVID-19                              988
Hypothyroidism                        803
Anal fissure                          766
Anemia                                747
Migraine                              731
Asthma                                692
Fatty Liver Disease                   685
Obesity                               684
B12 deficiency                        625
Anxiety                               498
Gallstone disease                     483
Irritable Bowel Syndrome (IBS)        475
Oral aphthae                          466
Hearing loss                      

In [18]:
baseline_conditions["research_stage"].value_counts().sort_index()

00_01_call         2
01_00_call      2589
01_01_visit        1
02_00_visit      130
02_01_call        14
03_00_call       154
04_00_visit       44
04_01_call        17
05_00_call        79
05_01_visit        1
06_00_visit       10
baseline       36613
Name: research_stage, dtype: int64

In [19]:
follow_up_conditions["research_stage"].value_counts().sort_index()

01_00_call     3664
01_01_call        1
02_00_visit    2134
02_01_call       59
03_00_call     2596
03_01_visit       2
03_02_call        1
04_00_visit     576
04_01_call       65
05_00_call      404
05_01_visit       2
06_00_visit      76
Name: research_stage, dtype: int64

In [20]:
def get_condition_matrix_for_stages(baseline_conditions, follow_up_conditions):
    research_stages = ['baseline', '02_00_visit', '04_00_visit']
    selected_conditions = [
        "Obesity",
        "Prediabetes",
        "Hyperlipidemia",
        "Hypertension",
        "Fatty Liver Disease (NAFLD)",
        "Irritable Bowel Syndrome (IBS)",
        "Peptic Ulcer Disease",
        "Gallstone disease",
        "Asthma",
        "Atopic dermatitis",
        "Psoriasis",
        "Allergy",
        "Depression",
        "Anxiety",
        "Hypothyroidism"
    ]
    cond_cols = ["RegistrationCode", "research_stage", "Consolidated name"]
    full = pd.concat([
        baseline_conditions.loc[:, cond_cols],
        follow_up_conditions.loc[:, cond_cols]
    ], ignore_index=True)

    output = {}
    for stage in research_stages:
        all_rcs_in_stage = pd.concat([
            baseline_conditions[baseline_conditions['research_stage'] == stage]['RegistrationCode'],
            follow_up_conditions[follow_up_conditions['research_stage'] == stage]['RegistrationCode'],
        ]).drop_duplicates().sort_values().tolist()

        matrix = pd.DataFrame(0, index=all_rcs_in_stage, columns=selected_conditions)

        stage_df = full[(full["research_stage"] == stage) & (full["Consolidated name"].isin(selected_conditions))]
        for row in stage_df.itertuples(index=False, name=None):
            rc = row[0]
            cond = row[2]
            if rc in matrix.index and cond in matrix.columns:
                matrix.at[rc, cond] = 1

        matrix = matrix.astype(int)
        output[stage] = matrix

    return output

condition_matrices = get_condition_matrix_for_stages(baseline_conditions, follow_up_conditions)
condition_matrices_baseline = condition_matrices['baseline']
condition_matrices_02_00_visit = condition_matrices['02_00_visit']
condition_matrices_04_00_visit = condition_matrices['04_00_visit']


In [21]:
# Helper to drop columns with no variation

def drop_no_variation(df: pd.DataFrame, name: str) -> pd.DataFrame:
    nunique = df.nunique()
    drop_cols = nunique[nunique <= 1].index.tolist()
    if drop_cols:
        print(f"Dropping {len(drop_cols)} no-variation columns from {name}: {drop_cols[:10]}")
        df = df.drop(columns=drop_cols)
    else:
        print(f"No no-variation columns in {name}")
    return df


In [22]:
# Drop comorbidity columns: remove from ALL timepoints only if zero variance at baseline
zero_var_baseline_cols = condition_matrices_baseline.columns[condition_matrices_baseline.nunique() <= 1].tolist()
if zero_var_baseline_cols:
    print(f"Comorbidity zero-variance at baseline: {zero_var_baseline_cols[:10]}")
    print(f"Dropping {len(zero_var_baseline_cols)} comorbidity columns (zero variance at baseline)")
    condition_matrices_baseline = condition_matrices_baseline.drop(columns=zero_var_baseline_cols, errors='ignore')
    condition_matrices_02_00_visit = condition_matrices_02_00_visit.drop(columns=zero_var_baseline_cols, errors='ignore')
    condition_matrices_04_00_visit = condition_matrices_04_00_visit.drop(columns=zero_var_baseline_cols, errors='ignore')
    print(f"Dropped: {zero_var_baseline_cols[:10]}")
else:
    print("No comorbidity columns dropped for zero variance at baseline")


Comorbidity zero-variance at baseline: ['Fatty Liver Disease (NAFLD)']
Dropping 1 comorbidity columns (zero variance at baseline)
Dropped: ['Fatty Liver Disease (NAFLD)']


In [23]:
condition_matrices_baseline

,Obesity,Prediabetes,Hyperlipidemia,Hypertension,Irritable Bowel Syndrome (IBS),Peptic Ulcer Disease,Gallstone disease,Asthma,Atopic dermatitis,Psoriasis,Allergy,Depression,Anxiety,Hypothyroidism
10K_1000942861,0,0,0,0,0,0,0,0,0,0,0,1,0,0
10K_1002033709,0,0,0,0,0,0,0,0,0,0,1,0,0,0
10K_1002087123,0,0,0,1,0,0,0,0,0,0,0,0,0,0
10K_1003113258,0,0,1,1,0,0,0,0,0,0,0,0,0,0
10K_1006172497,1,0,0,0,0,0,0,0,0,0,0,0,0,0
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
10K_9995746691,0,1,0,1,0,0,0,0,0,0,0,0,0,0
10K_9995823183,0,0,0,0,0,0,0,0,0,0,1,0,0,0
10K_9998418497,0,0,0,0,0,1,0,0,0,0,1,0,0,0
10K_9998635752,0,0,1,0,0,0,0,0,0,0,0,0,0,0


In [24]:
condition_matrices_02_00_visit

,Obesity,Prediabetes,Hyperlipidemia,Hypertension,Irritable Bowel Syndrome (IBS),Peptic Ulcer Disease,Gallstone disease,Asthma,Atopic dermatitis,Psoriasis,Allergy,Depression,Anxiety,Hypothyroidism
10K_1002087123,0,0,0,0,0,1,0,0,0,0,0,0,0,0
10K_1012400211,0,0,0,0,0,0,0,0,0,0,0,0,0,0
10K_1013508700,0,0,0,0,0,0,0,0,0,0,0,0,0,0
10K_1019174630,0,0,0,0,0,0,0,0,0,0,0,0,0,0
10K_1019625838,0,0,0,0,0,0,0,0,0,0,0,0,0,0
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
10K_9972023164,0,0,0,0,0,0,0,0,0,0,0,0,0,0
10K_9974013758,0,0,0,0,0,0,0,0,0,0,0,0,0,0
10K_9975645275,0,0,0,0,0,0,0,0,0,0,0,0,0,0
10K_9995623290,0,0,0,0,0,0,0,0,0,0,0,0,0,0


In [25]:
condition_matrices_04_00_visit

,Obesity,Prediabetes,Hyperlipidemia,Hypertension,Irritable Bowel Syndrome (IBS),Peptic Ulcer Disease,Gallstone disease,Asthma,Atopic dermatitis,Psoriasis,Allergy,Depression,Anxiety,Hypothyroidism
10K_1012020971,0,0,0,0,0,0,0,0,0,0,0,0,0,0
10K_1019625838,0,0,0,0,0,0,0,0,0,1,0,0,0,0
10K_1041215021,0,0,0,1,0,0,0,0,0,0,0,0,0,0
10K_1061997659,0,0,0,0,0,0,0,1,0,0,0,0,0,0
10K_1063317644,0,0,0,0,0,0,0,0,0,0,0,0,0,0
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
10K_9896364463,0,1,0,0,0,0,0,0,0,0,0,0,0,0
10K_9896598760,0,0,0,0,0,0,0,0,0,0,0,0,0,0
10K_9949352248,0,0,1,0,0,0,0,0,0,0,0,0,0,0
10K_9949511818,0,0,0,0,0,0,0,0,0,0,0,0,0,0


In [26]:
diet_mb = diet_mb.join(condition_matrices_baseline, how='left')
diet_mb = diet_mb.fillna(0)
diet_mb.shape


(10068, 1441)

### Lifestyle factors

In [27]:
dl = DemographicsLoader()
dld = dl.get_data(study_ids=study_ids, df="english")
dldf = dld.df
list(dldf.columns)

/home/tomerse/PycharmProjects/LabUtils/LabUtils/pandas_utils.py:101: FutureWarning: Inferring datetime64[ns, UTC] from data containing strings is deprecated and will be removed in a future version. To retain the old behavior explicitly pass Series(data, dtype={value.dtype})
  cond = df.index.isin(filter_df.index.unique())


['DOB',
 'army_position',
 'army_service',
 'assisted_living_current',
 'country_of_birth',
 'education',
 'education2',
 'education_complete_age',
 'employment',
 'employment_months',
 'employment_years',
 'father_country_of_birth',
 'gender',
 'grandfather_country_of_birth_father_side',
 'grandfather_country_of_birth_mother_side',
 'grandmother_country_of_birth_father_side',
 'grandmother_country_of_birth_mother_side',
 'living_place_today',
 'mother_country_of_birth',
 'profession',
 'total_income',
 'year_of_aliya',
 'years_of_army_service']

In [28]:
dldf.shape

(57002, 23)

In [29]:
# dldf[['employment', 'living_place_today']].describe()
# dldf['employment'].value_counts().sort_index()
dldf['living_place_today'].value_counts().sort_index()

City       22731
Moshav      3231
Other       2730
Village     1657
kibbutz     1338
Name: living_place_today, dtype: int64

In [30]:
dldf['living_place_today'].isna().sum()

25315

In [31]:
dldf

question_english                                    DOB army_position  \
RegistrationCode Date                                                   
10K_1000028368   2020-03-22 10:34:22               1966           NaN   
10K_1000633050   2021-06-21 17:36:02               1955           NaN   
10K_1000942861   2021-06-11 11:51:55               1967           NaN   
                 2023-03-20 08:01:31.157109+00:00   NaN           NaN   
                 2023-12-26 06:15:29.926993+00:00   NaN           NaN   
...                                                 ...           ...   
10K_9999226141   2025-11-19 10:27:07.216672+00:00   NaN           NaN   
10K_9999409119   2020-02-22 17:22:07               1966           NaN   
                 2023-06-06 11:27:36.711406+00:00   NaN           NaN   
10K_9999623844   2023-05-14 05:29:34.484000        1968           NaN   
10K_9999732920   2022-12-23 22:40:09.650000        1979           NaN   

question_english                                  army_service  \
RegistrationCode Date                                            
10K_1000028368   2020-03-22 10:34:22                       NaN   
10K_1000633050   2021-06-21 17:36:02                       NaN   
10K_1000942861   2021-06-11 11:51:55                       NaN   
                 2023-03-20 08:01:31.157109+00:00          NaN   
                 2023-12-26 06:15:29.926993+00:00          NaN   
...                                                        ...   
10K_9999226141   2025-11-19 10:27:07.216672+00:00          NaN   
10K_9999409119   2020-02-22 17:22:07                       NaN   
                 2023-06-06 11:27:36.711406+00:00          NaN   
10K_9999623844   2023-05-14 05:29:34.484000                NaN   
10K_9999732920   2022-12-23 22:40:09.650000                NaN   

question_english                                  assisted_living_current  \
RegistrationCode Date                                                       
10K_1000028368   2020-03-22 10:34:22                                  NaN   
10K_1000633050   2021-06-21 17:36:02                                  NaN   
10K_1000942861   2021-06-11 11:51:55                                  NaN   
                 2023-03-20 08:01:31.157109+00:00                     NaN   
                 2023-12-26 06:15:29.926993+00:00                     NaN   
...                                                                   ...   
10K_9999226141   2025-11-19 10:27:07.216672+00:00                     NaN   
10K_9999409119   2020-02-22 17:22:07                                  NaN   
                 2023-06-06 11:27:36.711406+00:00                     NaN   
10K_9999623844   2023-05-14 05:29:34.484000                           NaN   
10K_9999732920   2022-12-23 22:40:09.650000                           NaN   

question_english                                  country_of_birth  \
RegistrationCode Date                                                
10K_1000028368   2020-03-22 10:34:22                        Israel   
10K_1000633050   2021-06-21 17:36:02                        Israel   
10K_1000942861   2021-06-11 11:51:55                        Israel   
                 2023-03-20 08:01:31.157109+00:00              NaN   
                 2023-12-26 06:15:29.926993+00:00              NaN   
...                                                            ...   
10K_9999226141   2025-11-19 10:27:07.216672+00:00              NaN   
10K_9999409119   2020-02-22 17:22:07                        Israel   
                 2023-06-06 11:27:36.711406+00:00              NaN   
10K_9999623844   2023-05-14 05:29:34.484000         United Kingdom   
10K_9999732920   2022-12-23 22:40:09.650000                 Israel   

question_english                                           education  \
RegistrationCode Date                                                  
10K_1000028368   2020-03-22 10:34:22                 Master's degree   
10K_1000633050   2021-06-21 17:36:02                 Master's 

In [32]:
import pandas as pd

# Ensure dldf index includes both RegistrationCode and Date as a MultiIndex (if not, set it)
if not isinstance(dldf.index, pd.MultiIndex) or dldf.index.names != ['RegistrationCode', 'Date']:
    if 'RegistrationCode' in dldf.columns and 'Date' in dldf.columns:
        dldf = dldf.set_index(['RegistrationCode', 'Date'])
    elif 'Date' in dldf.columns:
        dldf = dldf.set_index(['Date'])
    else:
        # Assume index is already at least RegistrationCode, set Date as second if available
        if 'Date' in dldf.columns:
            dldf = dldf.set_index('Date', append=True)
        # If not available, leave as is (shouldn't happen for correctly loaded dldf)

dldf = dldf.copy()

# Robustly ensure the 'Date' index is datetime64, even with duplicate values, to prevent ValueError
if 'Date' in dldf.index.names and not pd.api.types.is_datetime64_any_dtype(dldf.index.get_level_values('Date')):
    # Convert Date level in MultiIndex to datetime using repeatable, non-unique-safe method
    # Rebuild MultiIndex with Date coerced to datetime on a value-by-value basis
    idx = dldf.index
    # Get current levels
    regcode_level = idx.get_level_values('RegistrationCode')
    date_level = pd.to_datetime(idx.get_level_values('Date'), errors='coerce')
    # Build new MultiIndex with same tuples, but Date forced to datetime
    dldf.index = pd.MultiIndex.from_arrays([regcode_level, date_level], names=['RegistrationCode', 'Date'])

baseline_rows = []
baseline_dates = {}

for regcode, group in dldf.groupby(level=0):
    group_sorted = group.sort_index(level=1)
    first_row = group_sorted.iloc[[0]]
    baseline_rows.append(first_row)
    baseline_dates[regcode] = group_sorted.index.get_level_values(1)[0]

demog_baseline = pd.concat(baseline_rows)
demog_baseline_dates = pd.Series(baseline_dates, name='Baseline_Date')

# Prepare lists for 2y and 4y visits
visit_02_rows = []
visit_04_rows = []

# Define window in days
six_months_days = 182
two_years_days = 2 * 365
four_years_days = 4 * 365

for regcode, group in dldf.groupby(level=0):
    if regcode not in baseline_dates:
        continue
    baseline_date = baseline_dates[regcode]
    # Use .loc trick to always drop the right indices, even if there are duplicate dates
    group_other = group.copy()
    # Remove all rows with baseline_date (may be more than one)
    if (group_other.index.get_level_values(1) == baseline_date).any():
        mask = group_other.index.get_level_values(1) != baseline_date
        group_other = group_other[mask]
    if group_other.empty:
        continue
    date_deltas = group_other.index.get_level_values(1) - baseline_date
    delta_days = date_deltas.days

    mask_02 = ((delta_days >= (two_years_days - six_months_days)) &
               (delta_days <= (two_years_days + six_months_days)))
    visit_02 = group_other[mask_02]
    if not visit_02.empty:
        visit_02_rows.append(visit_02)

    mask_04 = ((delta_days >= (four_years_days - six_months_days)) &
               (delta_days <= (four_years_days + six_months_days)))
    visit_04 = group_other[mask_04]
    if not visit_04.empty:
        visit_04_rows.append(visit_04)

if visit_02_rows:
    demog_02_visit = pd.concat(visit_02_rows)
else:
    demog_02_visit = pd.DataFrame(columns=dldf.columns)

if visit_04_rows:
    demog_04_visit = pd.concat(visit_04_rows)
else:
    demog_04_visit = pd.DataFrame(columns=dldf.columns)


In [33]:
demog_baseline = demog_baseline[["employment", "living_place_today", "total_income"]].reset_index(drop=True, level=1)
demog_02_visit = demog_02_visit[["employment", "living_place_today", "total_income"]].reset_index(drop=True, level=1)
demog_04_visit = demog_04_visit[["employment", "living_place_today", "total_income"]].reset_index(drop=True, level=1)

In [34]:
demog_02_visit

question_english,employment,living_place_today,total_income
RegistrationCode,,,
10K_1000942861,Retirement,City,"Over 36,000"
10K_1001201093,Employed or self-employed,City,"11,000 - 15,000"
10K_1002033709,Employed or self-employed,City,"Over 36,000"
10K_1002087123,Employed or self-employed,City,"6000 - 11,000"
10K_1002254441,Employed or self-employed,City,prefer not to answer
...,...,...,...
10K_9988216757,Employed or self-employed,City,"21,000 - 36,000"
10K_9991294748,Employed or self-employed,City,"6000 - 11,000"
10K_9995823183,Employed or self-employed,City,"Over 36,000"


In [35]:
print("Baseline duplicates:", demog_baseline.index.duplicated().any())
print("Visit 02 duplicates:", demog_02_visit.index.duplicated().any())
# Remove duplicates from the original DataFrames (keeping the first occurrence)
demog_baseline = demog_baseline.loc[~demog_baseline.index.duplicated(keep='first')]
demog_02_visit = demog_02_visit.loc[~demog_02_visit.index.duplicated(keep='first')]
demog_04_visit = demog_04_visit.loc[~demog_04_visit.index.duplicated(keep='first')]

Baseline duplicates: False
Visit 02 duplicates: True


In [36]:
common_ids = demog_baseline.index.intersection(demog_02_visit.index)
demog_02_visit_change = (demog_02_visit.loc[common_ids] != demog_baseline.loc[common_ids]).astype(int)

common_ids = demog_04_visit.index.intersection(demog_02_visit.index)
demog_04_visit_change = (demog_04_visit.loc[common_ids] != demog_02_visit.loc[common_ids]).astype(int)

# Drop columns with no variance across ALL demog change dataframes (baseline is all zeros)
combined_demog_changes = pd.concat([demog_02_visit_change, demog_04_visit_change], axis=0)
cols_drop_demog = combined_demog_changes.columns[combined_demog_changes.nunique() <= 1].tolist()
if cols_drop_demog:
    print(f"Dropping {len(cols_drop_demog)} no-variation demog change columns (across baseline/02/04): {cols_drop_demog[:10]}")
    demog_02_visit_change = demog_02_visit_change.drop(columns=cols_drop_demog)
    demog_04_visit_change = demog_04_visit_change.drop(columns=cols_drop_demog)
else:
    print("No demog change columns dropped for no variation across time points")

No demog change columns dropped for no variation across time points


In [37]:
print(demog_02_visit_change.shape)
print(demog_04_visit_change.shape)
demog_02_visit_change.tail()
demog_04_visit_change.tail()

(5245, 3)
(1793, 3)


question_english,employment,living_place_today,total_income
RegistrationCode,,,
10K_9972023164,0,0,0
10K_9982838880,1,0,1
10K_9985820154,0,0,0
10K_9995823183,0,0,0
10K_9999226141,1,0,1


In [38]:
demog_02_visit_change

question_english,employment,living_place_today,total_income
RegistrationCode,,,
10K_1000942861,1,1,1
10K_1001201093,1,1,1
10K_1002033709,1,1,1
10K_1002087123,1,1,1
10K_1002254441,1,1,1
...,...,...,...
10K_9988216757,1,1,1
10K_9991294748,1,1,1
10K_9995823183,1,1,1


In [39]:
print(demog_02_visit_change.sum(axis=0))
print(demog_02_visit_change.shape)
print(demog_04_visit_change.sum(axis=0))
print(demog_04_visit_change.shape)

question_english
employment            5155
living_place_today    5137
total_income          5170
dtype: int64
(5245, 3)
question_english
employment            273
living_place_today    147
total_income          728
dtype: int64
(1793, 3)


##### Lifestyle Loader

In [40]:
lll = LifeStyleLoader()
llld = lll.get_data(study_ids=study_ids)
llldf = llld.df
# list(llldf.columns)

In [41]:
list(llldf.columns)

['accommodation_type',
 'accommodation_years',
 'add_salt_to_food',
 'age_last_smoking_regularly1',
 'age_last_smoking_regularly_age',
 'alcohol_drink',
 'alcohol_drink_past',
 'beer_cider_pints_month',
 'beer_cider_pints_week',
 'bread_slices_week',
 'cereals_bowels_week',
 'cheese_fat_percentage_how',
 'cheese_milk_products',
 'cigaretts_last_age',
 'cigaretts_last_age_age',
 'cigaretts_past_per_day',
 'cigaretts_past_per_day_number1',
 'cigaretts_present_per_day',
 'climb_staires_tymes_a_day',
 'coffee_cups_day',
 'coffee_type',
 'consider_yourself_morning_evening',
 'cooked_veg_tablespoons_day',
 'diet_major_changes_5years',
 'diet_vary_week_to_week',
 'distance_from_home_to_work',
 'dried_fruit_day',
 'drink_alcohol_with_meals',
 'drink_compared_10years',
 'drive_faster_often',
 'easy_getting_up',
 'easy_go_without_smoking_day',
 'eat_beef',
 'eat_cereals_how',
 'eat_cheese',
 'eat_chicken_poultry',
 'eat_kosher',
 'eat_lamb_mutton',
 'eat_margarine',
 'eat_moldy_cheese_how',
 'ea

In [42]:
llldf

question_english                             accommodation_type  \
RegistrationCode Date                                             
10K_1000942861   2023-03-20 08:01:31.157109                 1.0   
                 2023-12-26 06:15:29.926993                 1.0   
10K_1001201093   2021-08-25 19:13:51.000000                 NaN   
                 2022-08-02 06:37:59.966886                 2.0   
                 2023-07-04 11:58:07.738595                 NaN   
...                                                         ...   
10K_9999226141   2023-03-19 09:04:11.612344                 1.0   
                 2024-01-15 16:20:17.350467                 1.0   
                 2025-11-19 10:27:07.216672                 1.0   
10K_9999409119   2020-03-08 16:16:34.000000                 2.0   
                 2023-06-06 11:27:36.711406                 2.0   

question_english                             accommodation_years  \
RegistrationCode Date                                              
10K_1000942861   2023-03-20 08:01:31.157109                 20.0   
                 2023-12-26 06:15:29.926993                 22.0   
10K_1001201093   2021-08-25 19:13:51.000000                  NaN   
                 2022-08-02 06:37:59.966886                 15.0   
                 2023-07-04 11:58:07.738595                 15.0   
...                                                          ...   
10K_9999226141   2023-03-19 09:04:11.612344                  4.0   
                 2024-01-15 16:20:17.350467                  5.0   
                 2025-11-19 10:27:07.216672                  6.0   
10K_9999409119   2020-03-08 16:16:34.000000                 11.0   
                 2023-06-06 11:27:36.711406                 12.0   

question_english                             add_salt_to_food  \
RegistrationCode Date                                           
10K_1000942861   2023-03-20 08:01:31.157109               NaN   
                 2023-12-26 06:15:29.926993               NaN   
10K_1001201093   2021-08-25 19:13:51.000000               NaN   
                 2022-08-02 06:37:59.966886               NaN   
                 2023-07-04 11:58:07.738595               NaN   
...                                                       ...   
10K_9999226141   2023-03-19 09:04:11.612344               NaN   
                 2024-01-15 16:20:17.350467               0.5   
                 2025-11-19 10:27:07.216672               NaN   
10K_9999409119   2020-03-08 16:16:34.000000               NaN   
                 2023-06-06 11:27:36.711406               NaN   

question_english                             age_last_smoking_regularly1  \
RegistrationCode Date                                                      
10K_1000942861   2023-03-20 08:01:31.157109                          NaN   
                 2023-12-26 06:15:29.926993                          NaN   
10K_1001201093   2021-08-25 19:13:51.000000                          NaN   
                 2022-08-02 06:37:59.966886                          NaN   
                 2023-07-04 11:58:07.738595                          NaN   
...                                                                  ...   
10K_9999226141   2023-03-19 09:04:11.612344                          NaN   
                 2024-01-15 16:20:17.350467                          NaN   
                 2025-11-19 10:27:07.216672                          NaN   
10K_9999409119   2020-03-08 16:16:34.000000                          NaN   
                 2023-06-06 11:27:36.711406                          NaN   

question_english                             age_last_smoking_regularly_age  \
RegistrationCode Date                                                         
10K_1000942861   2023-03-20 08:01:31.157109                             NaN   
                 2023-12-26 06:15:29.926993                             NaN   
10K_1001201093   2021-08-25 19:13:51.000000                             NaN   
             

In [43]:
import pandas as pd

# Assume llldf is already loaded and has MultiIndex (RegistrationCode, Date)
# Ensure Date index is datetime
llldf = llldf.copy()
if not pd.api.types.is_datetime64_any_dtype(llldf.index.get_level_values("Date")):
    llldf.index = llldf.index.set_levels(
        pd.to_datetime(llldf.index.levels[1]), level=1
    )

# Find baseline (first) date for each RegistrationCode
baseline_rows = []
baseline_dates = {}

for regcode, group in llldf.groupby(level=0):
    group_sorted = group.sort_index(level=1)
    first_row = group_sorted.iloc[[0]]
    baseline_rows.append(first_row)
    baseline_dates[regcode] = group_sorted.index.get_level_values(1)[0]

lifestyle_baseline = pd.concat(baseline_rows)
baseline_date_series = pd.Series(baseline_dates, name="Baseline_Date")

# Prepare empty lists to collect 02 and 04 visit rows
visit_02_rows = []
visit_04_rows = []

# Define window (in days)
six_months_days = 182  # ~6 months
two_years_days = 2 * 365
four_years_days = 4 * 365

for regcode, group in llldf.groupby(level=0):
    if regcode not in baseline_dates:
        continue  # shouldn't happen
    baseline_date = baseline_dates[regcode]
    # Exclude baseline row itself (by date)
    group_other = group.drop(baseline_date, level=1, errors="ignore")
    if group_other.empty:
        continue
    # Compute delta days for each row from baseline
    date_deltas = group_other.index.get_level_values(1) - baseline_date
    delta_days = date_deltas.days

    # Find rows within 2y +/-6m (i.e., [2y-6m, 2y+6m])
    mask_02 = ((delta_days >= (two_years_days - six_months_days)) &
               (delta_days <= (two_years_days + six_months_days)))
    visit_02 = group_other[mask_02]
    if not visit_02.empty:
        visit_02_rows.append(visit_02)

    # Find rows within 4y +/-6m
    mask_04 = ((delta_days >= (four_years_days - six_months_days)) &
               (delta_days <= (four_years_days + six_months_days)))
    visit_04 = group_other[mask_04]
    if not visit_04.empty:
        visit_04_rows.append(visit_04)

if visit_02_rows:
    lifestyle_02_visit = pd.concat(visit_02_rows)
else:
    lifestyle_02_visit = pd.DataFrame(columns=llldf.columns)

if visit_04_rows:
    lifestyle_04_visit = pd.concat(visit_04_rows)
else:
    lifestyle_04_visit = pd.DataFrame(columns=llldf.columns)

# Reset index for baseline for consistency (optional), or keep as MultiIndex if preferred
lifestyle_baseline = lifestyle_baseline.copy()



In [44]:
lifestyle_baseline

,question_english,accommodation_type,accommodation_years,add_salt_to_food,age_last_smoking_regularly1,age_last_smoking_regularly_age,alcohol_drink,alcohol_drink_past,beer_cider_pints_month,beer_cider_pints_week,bread_slices_week,...,transportaion_to_work__Cab,transportaion_to_work__Car / motorcycle,transportaion_to_work__Electric Bicycle,transportaion_to_work__Electric scooter,transportaion_to_work__Train,transportaion_to_work__Walk,transportaion_to_work__Walking,transportaion_to_work__no,transportaion_to_work__prefer not to answer,"transportaion_to_work__אף אחד מהנ""ל"
RegistrationCode,Date,,,,,,,,,,,,,,,,,,,,,
10K_1000942861,2023-03-20 08:01:31.157109,1.0,20.0,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,0.0,1.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
10K_1001201093,2021-08-25 19:13:51.000000,NaN,NaN,NaN,NaN,NaN,1.0,NaN,NaN,0.0,7.0,...,0.0,1.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
10K_1002033709,2022-05-31 08:30:09.969853,2.0,1.0,NaN,NaN,NaN,3.0,NaN,NaN,1.0,7.0,...,0.0,0.0,0.0,0.0,0.0,0.0,1.0,0.0,0.0,0.0
10K_1002087123,2019-11-11 10:23:30.000000,2.0,4.0,3.0,NaN,36.0,2.0,NaN,NaN,0.0,10.0,...,0.0,1.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
10K_1002254441,2022-08-29 19:46:11.531119,NaN,1.0,NaN,NaN,NaN,NaN,1.0,0.0,0.0,NaN,...,0.0,1.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
10K_9996884777,2021-11-25 18:57:07.990147,1.0,21.0,1.0,NaN,NaN,3.0,NaN,NaN,0.0,NaN,...,0.0,1.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
10K_9998420917,2023-02-23 11:32:13.755163,1.0,10.0,1.0,NaN,25.0,2.0,NaN,-1.0,-1.0,10.0,...,0.0,1.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
10K_9998635752,2024-04-07 16:20:43.184388,1.0,3.0,NaN,NaN,NaN,2.0,NaN,4.0,1.0,10.0,...,0.0,1.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0


In [45]:
print(lifestyle_02_visit.shape)
print(lifestyle_04_visit.shape)


(5180, 262)
(2715, 262)


In [46]:
def process_lifestyle_data(lifestyle_df: pd.DataFrame) -> pd.DataFrame:
    """
    Processes the lifestyle DataFrame:
    - Removes the 'Date' index, keeping only 'RegistrationCode'
    - Selects several direct columns
    - Aggregates smoking columns into binary columns
    - Aggregates household relationship columns into grouped binary columns
    - Normalizes all -1 values to 0
    - Fills NaN values in pet_present and accommodation_type with 0

    Args:
        lifestyle_df: The input DataFrame with MultiIndex ('RegistrationCode', 'Date')

    Returns:
        A new DataFrame with processed columns, indexed by RegistrationCode
    """
    # Remove 'Date' index, keep only RegistrationCode
    if isinstance(lifestyle_df.index, pd.MultiIndex):
        lifestyle_df = lifestyle_df.reset_index(level='Date', drop=True)

    result_cols = [
        'pet_present',
        'accommodation_type',
    ]
    df_result = lifestyle_df[result_cols].copy()

    # Fill pet_present and accommodation_type NaNs with 0
    for col in ['pet_present', 'accommodation_type']:
        if col in df_result.columns:
            df_result[col] = df_result[col].fillna(0)

    # --- Smoking columns ---
    current_smoker_col = 'smoke_tobacco_now'
    if current_smoker_col in lifestyle_df.columns:
        df_result['is_current_smoker'] = lifestyle_df[current_smoker_col].fillna(0).astype(int)
    else:
        df_result['is_current_smoker'] = 0

    household_smoker_col = 'smoke_houshold'
    if household_smoker_col in lifestyle_df.columns:
        df_result['is_household_smoker'] = lifestyle_df[household_smoker_col].fillna(0).astype(int)
    else:
        df_result['is_household_smoker'] = 0

    # --- Relationship columns aggregation ---
    living_with_relative_cols = [
        'people_living_together_retalated__Another connection',
        'people_living_together_retalated__Brother, sister or both',
        'people_living_together_retalated__Grandchildren',
        'people_living_together_retalated__Grandparents',
        'people_living_together_retalated__Son, daughter or both',
        'people_living_together_retalated__parents'
    ]
    living_with_relative_cols_present = [c for c in living_with_relative_cols if c in lifestyle_df.columns]
    if living_with_relative_cols_present:
        df_result['living_with_relative'] = lifestyle_df[living_with_relative_cols_present].fillna(0).max(axis=1).astype(int)
    else:
        df_result['living_with_relative'] = 0

    # Spouse/partner (treated separately from other unrelated)
    living_with_spouse_cols = [
        'people_living_together_retalated__Husband, wife or spouse',
        'people_living_together_retalated__Partner'
    ]
    living_with_spouse_cols_present = [c for c in living_with_spouse_cols if c in lifestyle_df.columns]
    if living_with_spouse_cols_present:
        df_result['living_with_spouse'] = lifestyle_df[living_with_spouse_cols_present].fillna(0).max(axis=1).astype(int)
    else:
        df_result['living_with_spouse'] = 0

    # Other not-related (excluding spouse/partner)
    living_with_not_related_cols = [
        'people_living_together_retalated__Other',
        'people_living_together_retalated__Other unrelated'
    ]
    living_with_not_related_cols_present = [c for c in living_with_not_related_cols if c in lifestyle_df.columns]
    if living_with_not_related_cols_present:
        df_result['living_with_not_related'] = lifestyle_df[living_with_not_related_cols_present].fillna(0).max(axis=1).astype(int)
    else:
        df_result['living_with_not_related'] = 0

    # Normalize -1 to 0 across all columns
    df_result = df_result.replace(-1, 0)

    return df_result

lifestyle_baseline = process_lifestyle_data(lifestyle_baseline)
print(lifestyle_baseline.head())

question_english  pet_present  accommodation_type  is_current_smoker  \
RegistrationCode                                                       
10K_1000942861            0.0                 1.0                  0   
10K_1001201093            0.0                 0.0                  0   
10K_1002033709            0.0                 2.0                  0   
10K_1002087123            1.0                 2.0                  0   
10K_1002254441            0.0                 0.0                  0   

question_english  is_household_smoker  living_with_relative  \
RegistrationCode                                              
10K_1000942861                      0                     1   
10K_1001201093                      0                     0   
10K_1002033709                      0                     1   
10K_1002087123                      0                     0   
10K_1002254441                      0                     1   

question_english  living_with_spouse  living_with_not

In [47]:
lifestyle_02_visit = process_lifestyle_data(lifestyle_02_visit)
print(lifestyle_02_visit.head())
lifestyle_04_visit = process_lifestyle_data(lifestyle_04_visit)
print(lifestyle_04_visit.head())

question_english  pet_present  accommodation_type  is_current_smoker  \
RegistrationCode                                                       
10K_1001201093            0.0                 0.0                  0   
10K_1002087123            1.0                 2.0                  0   
10K_1002254441            0.0                 2.0                  0   
10K_1007400622            1.0                 1.0                  0   
10K_1008294272            0.0                 2.0                  0   

question_english  is_household_smoker  living_with_relative  \
RegistrationCode                                              
10K_1001201093                      0                     0   
10K_1002087123                      0                     0   
10K_1002254441                      0                     1   
10K_1007400622                      0                     1   
10K_1008294272                      0                     1   

question_english  living_with_spouse  living_with_not

In [48]:
lifestyle_04_visit.isna().sum()
lifestyle_02_visit.isna().sum()

question_english
pet_present                0
accommodation_type         0
is_current_smoker          0
is_household_smoker        0
living_with_relative       0
living_with_spouse         0
living_with_not_related    0
dtype: int64

In [49]:
for col in lifestyle_baseline.columns:
    print(col)
    print(lifestyle_baseline[col].value_counts())
    print('-'*100)

pet_present
0.0    7404
1.0    5474
Name: pet_present, dtype: int64
----------------------------------------------------------------------------------------------------
accommodation_type
2.0    5984
1.0    5556
0.0    1303
3.0      28
4.0       5
5.0       2
Name: accommodation_type, dtype: int64
----------------------------------------------------------------------------------------------------
is_current_smoker
0    12160
1      718
Name: is_current_smoker, dtype: int64
----------------------------------------------------------------------------------------------------
is_household_smoker
0    11835
1      897
2      146
Name: is_household_smoker, dtype: int64
----------------------------------------------------------------------------------------------------
living_with_relative
1    9725
0    3153
Name: living_with_relative, dtype: int64
----------------------------------------------------------------------------------------------------
living_with_spouse
1    10666
0     2212
Nam

In [50]:
print(lifestyle_baseline.info())
print(lifestyle_02_visit.info())
print(lifestyle_04_visit.info())


<class 'pandas.core.frame.DataFrame'>
Index: 12878 entries, 10K_1000942861 to 10K_9999409119
Data columns (total 7 columns):
 #   Column                   Non-Null Count  Dtype  
---  ------                   --------------  -----  
 0   pet_present              12878 non-null  float64
 1   accommodation_type       12878 non-null  float64
 2   is_current_smoker        12878 non-null  int64  
 3   is_household_smoker      12878 non-null  int64  
 4   living_with_relative     12878 non-null  int64  
 5   living_with_spouse       12878 non-null  int64  
 6   living_with_not_related  12878 non-null  int64  
dtypes: float64(2), int64(5)
memory usage: 804.9+ KB
None
<class 'pandas.core.frame.DataFrame'>
Index: 5180 entries, 10K_1001201093 to 10K_9999226141
Data columns (total 7 columns):
 #   Column                   Non-Null Count  Dtype  
---  ------                   --------------  -----  
 0   pet_present              5180 non-null   float64
 1   accommodation_type       5180 non-null  

In [51]:
print("Baseline duplicates:", lifestyle_baseline.index.duplicated().any())
print("Visit 02 duplicates:", lifestyle_02_visit.index.duplicated().any())

Baseline duplicates: False
Visit 02 duplicates: True


In [52]:
# Remove duplicates from the original DataFrames (keeping the first occurrence)
lifestyle_baseline = lifestyle_baseline.loc[~lifestyle_baseline.index.duplicated(keep='first')]
lifestyle_02_visit = lifestyle_02_visit.loc[~lifestyle_02_visit.index.duplicated(keep='first')]
lifestyle_04_visit = lifestyle_04_visit.loc[~lifestyle_04_visit.index.duplicated(keep='first')]

In [53]:
common_ids = lifestyle_baseline.index.intersection(lifestyle_02_visit.index)
lifestyle_02_visit_change = (lifestyle_02_visit.loc[common_ids] != lifestyle_baseline.loc[common_ids]).astype(int)
lifestyle_02_visit_change = drop_no_variation(lifestyle_02_visit_change, "lifestyle_02_visit_change")

common_ids = lifestyle_04_visit.index.intersection(lifestyle_02_visit.index)
lifestyle_04_visit_change = (lifestyle_04_visit.loc[common_ids] != lifestyle_02_visit.loc[common_ids]).astype(int)
lifestyle_04_visit_change = drop_no_variation(lifestyle_04_visit_change, "lifestyle_04_visit_change")

No no-variation columns in lifestyle_02_visit_change
No no-variation columns in lifestyle_04_visit_change


In [54]:
print(lifestyle_02_visit_change.shape)
print(lifestyle_04_visit_change.shape)

(5100, 7)
(1709, 7)


In [55]:
lifestyle_04_visit_change.tail()

question_english,pet_present,accommodation_type,is_current_smoker,is_household_smoker,living_with_relative,living_with_spouse,living_with_not_related
RegistrationCode,,,,,,,
10K_9970701406,0,0,0,0,0,0,0
10K_9971776707,0,0,0,0,0,0,0
10K_9972023164,0,0,0,0,0,0,0
10K_9982838880,0,0,0,0,0,0,0
10K_9999226141,0,0,0,1,0,1,0


In [56]:
print(lifestyle_02_visit_change.sum(axis=0))
print(lifestyle_02_visit_change.shape)
print(lifestyle_04_visit_change.sum(axis=0))
print(lifestyle_04_visit_change.shape)

question_english
pet_present                 979
accommodation_type         1192
is_current_smoker           160
is_household_smoker         517
living_with_relative        543
living_with_spouse          207
living_with_not_related     128
dtype: int64
(5100, 7)
question_english
pet_present                319
accommodation_type         467
is_current_smoker           44
is_household_smoker        186
living_with_relative       182
living_with_spouse          77
living_with_not_related     30
dtype: int64
(1709, 7)


#### Adam and Gil's datasets

In [57]:
# lifestyle_baseline = pd.read_csv("/net/mraid20/ifs/wisdom/segal_lab/genie/LabData/Analyses/10K_Trajectories/body_systems/lifestyle_baseline.csv")
# lifestyle_02_visit = pd.read_csv("/net/mraid20/ifs/wisdom/segal_lab/genie/LabData/Analyses/10K_Trajectories/body_systems/lifestyle_02_00_visit.csv")
# lifestyle_04_visit = pd.read_csv("/net/mraid20/ifs/wisdom/segal_lab/genie/LabData/Analyses/10K_Trajectories/body_systems/lifestyle_04_00_visit.csv")
# lifestyle_baseline

In [58]:
# def process_lifestyle_data(lifestyle_df: pd.DataFrame) -> pd.DataFrame:
#     """
#     Selects 'pet_present_yes' and combines smoking-related columns 
#     into two new binary columns. Combination means 1 if any component is 1, else 0.

#     Args:
#         lifestyle_df: The input DataFrame containing the lifestyle columns.

#     Returns:
#         A new DataFrame with the three processed columns.
#     """
    
#     # 1. Select the direct column
#     lifestyle_df.set_index('RegistrationCode', inplace=True)
#     df_result = lifestyle_df[['pet_present_yes']].copy()
    
#     # 2. Combine columns for person currently smokes
#     current_smoker_cols = [
#         'smoke_tobacco_now_Only sometimes', 
#         'smoke_tobacco_now_Yes  most or all days'
#     ]
    
#     # Use max(axis=1) for the OR operation: 1 if any column is 1
#     df_result['is_current_smoker'] = lifestyle_df[current_smoker_cols].max(axis=1)
    
#     # 3. Combine columns for someone in the household smokes
#     household_smoker_cols = [
#         'smoke_houshold_Yes  more than one household member smokes', 
#         'smoke_houshold_Yes  one household member smokes'
#     ]
    
#     # Use max(axis=1) for the OR operation: 1 if any column is 1
#     df_result['is_household_smoker'] = lifestyle_df[household_smoker_cols].max(axis=1)
    
#     return df_result


# lifestyle_baseline = process_lifestyle_data(lifestyle_baseline)
# print(lifestyle_baseline.head())

In [59]:
# lifestyle_02_visit = process_lifestyle_data(lifestyle_02_visit)
# print(lifestyle_02_visit.head())
# lifestyle_04_visit = process_lifestyle_data(lifestyle_04_visit)
# print(lifestyle_04_visit.head())

In [60]:
# lifestyle_04_visit.isna().sum()
# lifestyle_02_visit.isna().sum()

In [61]:
# common_ids = lifestyle_baseline.index.intersection(lifestyle_02_visit.index)
# lifestyle_02_visit_change = lifestyle_02_visit.loc[common_ids] - lifestyle_baseline.loc[common_ids]
# common_ids = lifestyle_04_visit.index.intersection(lifestyle_02_visit.index)
# lifestyle_04_visit_change = lifestyle_04_visit.loc[common_ids] - lifestyle_02_visit.loc[common_ids]

In [62]:
# lifestyle_02_visit_change.shape

### Medications

In [63]:
# # Load medications data
# ml = Medications10KLoader()
# mld = ml.get_data(study_ids=study_ids, pivot_by=3)
# mldf = mld.df
# list(mldf.columns)

In [64]:
medications_baseline = pd.read_csv("/net/mraid20/ifs/wisdom/segal_lab/genie/LabData/Analyses/10K_Trajectories/body_systems/medications_baseline.csv")
medications_02_visit = pd.read_csv("/net/mraid20/ifs/wisdom/segal_lab/genie/LabData/Analyses/10K_Trajectories/body_systems/medications_02_00_visit.csv")
medications_04_visit = pd.read_csv("/net/mraid20/ifs/wisdom/segal_lab/genie/LabData/Analyses/10K_Trajectories/body_systems/medications_04_00_visit.csv")
medications_baseline

,RegistrationCode,research_stage,ACEINHIBITORSCOMBINATIONS,ACEINHIBITORSPLAIN,ADRENERGICSINHALANTS,ANGIOTENSINIIRECEPTORBLOCKERSARBsCOMBINATIONS,ANGIOTENSINIIRECEPTORBLOCKERSARBsPLAIN,ANTIADRENERGICAGENTSPERIPHERALLYACTING,ANTIDEPRESSANTS,ANTIEPILEPTICS,...,PROGESTOGENSEXHORMONESANDMODULATORSOFTHEGENITALSYSTEM,PROGESTOGENSANDESTROGENSINCOMBINATION,PSYCHOSTIMULANTSAGENTSUSEDFORADHDANDNOOTROPICS,SELECTIVECALCIUMCHANNELBLOCKERSWITHMAINLYVASCULAREFFECTS,THYROIDPREPARATIONS,UROLOGICALS,VITAMINAANDDINCLCOMBINATIONSOFTHETWO,VITAMINBn12nANDFOLICACID,VITAMINBn1nPLAINANDINCOMBINATIONWITHVITAMINBn6nANDBn12n,VITAMINKANDOTHERHEMOSTATICS
0,10K_1000942861,baseline,0.0,0.0,0.0,0.0,0.0,0.0,1.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
1,10K_1001201093,baseline,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
2,10K_1002033709,baseline,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
3,10K_1002087123,baseline,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
4,10K_1002254441,baseline,0.0,0.0,0.0,0.0,0.0,0.0,1.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
13486,10K_9998418497,baseline,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
13487,10K_9998420917,baseline,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
13488,10K_9998635752,baseline,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
13489,10K_9999226141,baseline,0.0,0.0,0.0,0.0,0.0,0.0,1.0,0.0,...,0.0,0.0,1.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0


In [65]:
# Ensure lifestyle change columns have variation across all time points (baseline is zeros)
combined_lifestyle_changes = pd.concat([lifestyle_02_visit_change, lifestyle_04_visit_change], axis=0)
cols_drop_lifestyle = combined_lifestyle_changes.columns[combined_lifestyle_changes.nunique() <= 1].tolist()
if cols_drop_lifestyle:
    print(f"Dropping {len(cols_drop_lifestyle)} no-variation lifestyle change columns (across baseline/02/04): {cols_drop_lifestyle[:10]}")
    lifestyle_02_visit_change = lifestyle_02_visit_change.drop(columns=cols_drop_lifestyle)
    lifestyle_04_visit_change = lifestyle_04_visit_change.drop(columns=cols_drop_lifestyle)
else:
    print("No lifestyle change columns dropped for no variation across time points")


No lifestyle change columns dropped for no variation across time points


In [66]:
# Find columns (medications) present in all three DataFrames
common_columns = set(medications_baseline.columns) & set(medications_02_visit.columns) & set(medications_04_visit.columns)
common_columns = list(common_columns)

# Subset the DataFrames to these columns
mb = medications_baseline[common_columns]
m2 = medications_02_visit[common_columns]
m4 = medications_04_visit[common_columns]

# Compute prevalence for each medication in each dataset
prevalence_baseline = mb.mean()
prevalence_02_visit = m2.mean()
prevalence_04_visit = m4.mean()

valid_range_baseline = (prevalence_baseline > 0.05) & (prevalence_baseline < 0.95)
valid_range_02_visit = (prevalence_02_visit > 0.05) & (prevalence_02_visit < 0.95)
valid_range_04_visit = (prevalence_04_visit > 0.05) & (prevalence_04_visit < 0.95)

# Find columns with True in all three datasets
cols_in_valid_range_all = valid_range_baseline & valid_range_02_visit & valid_range_04_visit
cols_in_valid_range_all = cols_in_valid_range_all[cols_in_valid_range_all].index.tolist()

print("Columns with prevalence in (0.05, 0.95) for all three datasets:", cols_in_valid_range_all)

Columns with prevalence in (0.05, 0.95) for all three datasets: ['LIPIDMODIFYINGAGENTSPLAIN']


/usr/wisdom/python3/lib/python3.7/site-packages/ipykernel_launcher.py:11: FutureWarning: Dropping of nuisance columns in DataFrame reductions (with 'numeric_only=None') is deprecated; in a future version this will raise TypeError.  Select only valid columns before calling the reduction.
  # This is added back by InteractiveShellApp.init_path()
/usr/wisdom/python3/lib/python3.7/site-packages/ipykernel_launcher.py:12: FutureWarning: Dropping of nuisance columns in DataFrame reductions (with 'numeric_only=None') is deprecated; in a future version this will raise TypeError.  Select only valid columns before calling the reduction.
  if sys.path[0] == '':
/usr/wisdom/python3/lib/python3.7/site-packages/ipykernel_launcher.py:13: FutureWarning: Dropping of nuisance columns in DataFrame reductions (with 'numeric_only=None') is deprecated; in a future version this will raise TypeError.  Select only valid columns before calling the reduction.
  del sys.path[0]


In [67]:
print((prevalence_baseline > 0.05) & (prevalence_baseline < 0.95))
print((prevalence_02_visit > 0.05) & (prevalence_02_visit < 0.95))
print((prevalence_04_visit > 0.05) & (prevalence_04_visit < 0.95))


BETABLOCKINGAGENTS                                          False
ESTROGENSSEXHORMONESANDMODULATORSOFTHEGENITALSYSTEM         False
OTHERSEXHORMONESANDMODULATORSOFTHEGENITALSYSTEMinATC        False
IRONANTIANEMICPREPARATIONS                                  False
DRUGSUSEDINBENIGNPROSTATICHYPERTROPHY                       False
HORMONALCONTRACEPTIVESFORSYSTEMICUSE                        False
LIPIDMODIFYINGAGENTSPLAIN                                    True
CALCIUMSUPPLEMENTS                                          False
ANTIGLAUCOMAPREPARATIONSANDMIOTICS                          False
ANTIINFLAMMATORYANDANTIRHEUMATICPRODUCTSNONSTEROIDS         False
VITAMINKANDOTHERHEMOSTATICS                                 False
THYROIDPREPARATIONS                                         False
HORMONEANTAGONISTSANDRELATEDAGENTS                          False
ANTIADRENERGICAGENTSPERIPHERALLYACTING                      False
PROGESTOGENSANDESTROGENSINCOMBINATION                       False
OTHERANALG

In [68]:
# For each medications dataset, subset to cols_in_valid_range_all and set RegistrationCode as index
mb_sub = medications_baseline[['RegistrationCode'] + cols_in_valid_range_all].set_index('RegistrationCode')
m2_sub = medications_02_visit[['RegistrationCode'] + cols_in_valid_range_all].set_index('RegistrationCode')
m4_sub = medications_04_visit[['RegistrationCode'] + cols_in_valid_range_all].set_index('RegistrationCode')


In [69]:
# For each subject, create a DataFrame showing medication changes between visits
# Change from baseline to 02 visit
common_ids_b2 = mb_sub.index.intersection(m2_sub.index)
medications_02_visit_change = (m2_sub.loc[common_ids_b2] != mb_sub.loc[common_ids_b2]).astype(int)

# Change from 02 to 04 visit
common_ids_24 = m2_sub.index.intersection(m4_sub.index)
medications_04_visit_change = (m4_sub.loc[common_ids_24] != m2_sub.loc[common_ids_24]).astype(int)

# Drop columns with no variance across ALL medication change dataframes (baseline is all zeros)
combined_med_changes = pd.concat([medications_02_visit_change, medications_04_visit_change], axis=0)
cols_drop_med = combined_med_changes.columns[combined_med_changes.nunique() <= 1].tolist()
if cols_drop_med:
    print(f"Dropping {len(cols_drop_med)} no-variation medications change columns (across baseline/02/04): {cols_drop_med[:10]}")
    medications_02_visit_change = medications_02_visit_change.drop(columns=cols_drop_med)
    medications_04_visit_change = medications_04_visit_change.drop(columns=cols_drop_med)
else:
    print("No medications change columns dropped for no variation across time points")


No medications change columns dropped for no variation across time points


In [70]:
medications_02_visit_change

,LIPIDMODIFYINGAGENTSPLAIN
RegistrationCode,
10K_1001201093,0
10K_1002087123,0
10K_1002254441,0
10K_1007400622,0
10K_1007474664,0
...,...
10K_9995623290,0
10K_9995746691,0
10K_9995823183,0


In [71]:
medications_04_visit_change

,LIPIDMODIFYINGAGENTSPLAIN
RegistrationCode,
10K_1001201093,0
10K_1007599726,0
10K_1007699078,1
10K_1012020971,0
10K_1019625838,1
...,...
10K_9977346258,1
10K_9982838880,0
10K_9984191583,0


In [72]:
# # Create a version of diet_mb with Fatty Liver Disease (NAFLD) column included (filled with zeros)
# # This is for a colleague who needs the column even though it has zero variance at baseline
# diet_mb_with_fatty_liver = pd.read_pickle(home_path + f"data/{SPECIES}/diet_mb.pkl")

# # Add Fatty Liver Disease (NAFLD) column with all zeros
# fatty_liver_col_name = "Fatty Liver Disease (NAFLD)"
# diet_mb_with_fatty_liver[fatty_liver_col_name] = 0

# # Save with a new name
# output_path = home_path + f"data/{SPECIES}/diet_mb_with_fatty_liver.pkl"
# diet_mb_with_fatty_liver.to_pickle(output_path)
# print(f"Saved diet_mb with Fatty Liver column to: {output_path}")
# print(f"Shape: {diet_mb_with_fatty_liver.shape}")
# print(f"Fatty Liver column present: {fatty_liver_col_name in diet_mb_with_fatty_liver.columns}")


### Saving

In [73]:
bmi = "bmi"
comorbidities = condition_matrices_baseline.columns
lifestyle = lifestyle_baseline.columns

In [74]:
# Add covariates to 02/04 visit datasets and save
# Read visit datasets similar to train_models.py
pathways = ''  # keep empty to mirror current files
CLR_suf = ''   # no CLR suffix in this notebook

# Load visit diet_mb tables
visit_02_path = home_path + f"data/{SPECIES}/diet_mb{pathways}_02_visit{CLR_suf}.pkl"
visit_04_path = home_path + f"data/{SPECIES}/diet_mb{pathways}_04_visit{CLR_suf}.pkl"
diet_mb_02_visit = pd.read_pickle(visit_02_path)
diet_mb_04_visit = pd.read_pickle(visit_04_path)

# Helper to select BMI per stage (same logic as baseline block above)
def select_bmi_for_stage(df, stage_label):
    if stage_label == 'baseline':
        out = df[~df.index.get_level_values(0).duplicated()]
        out = out.reset_index(level=[1], drop=True)
    elif stage_label == '02_00_visit':
        out = df.groupby(level=0).nth(1)
    elif stage_label == '04_00_visit':
        out = df.groupby(level=0).nth(1)
    else:
        out = df
    return out['bmi']

# Reuse the original body measures dataframe
bmldf_full = bml.get_data(study_ids=study_ids).df
bmi_02 = select_bmi_for_stage(bmldf_full.copy(), '02_00_visit')
bmi_04 = select_bmi_for_stage(bmldf_full.copy(), '04_00_visit')

# Join BMI
diet_mb_02_visit = diet_mb_02_visit.join(bmi_02, how='left')
diet_mb_04_visit = diet_mb_04_visit.join(bmi_04, how='left')

# Join comorbidities for the matching stage and fill missing with 0
if 'condition_matrices_02_00_visit' in globals():
    diet_mb_02_visit = diet_mb_02_visit.join(condition_matrices_02_00_visit, how='left')
    diet_mb_02_visit = diet_mb_02_visit.fillna(0)

if 'condition_matrices_04_00_visit' in globals():
    diet_mb_04_visit = diet_mb_04_visit.join(condition_matrices_04_00_visit, how='left')
    diet_mb_04_visit = diet_mb_04_visit.fillna(0)

# Save the visit datasets with covariates
visit_02_out = home_path + f"data/{SPECIES}/diet_mb{pathways}_02_visit{CLR_suf}.pkl"
visit_04_out = home_path + f"data/{SPECIES}/diet_mb{pathways}_04_visit{CLR_suf}.pkl"
diet_mb_02_visit.to_pickle(visit_02_out)
diet_mb_04_visit.to_pickle(visit_04_out)

print("Saved:", visit_02_out)
print("Saved:", visit_04_out)


Saved: /net/mraid20/export/genie/LabData/Analyses/tomerse/diet_mb/data/segal_species/diet_mb_02_visit.pkl
Saved: /net/mraid20/export/genie/LabData/Analyses/tomerse/diet_mb/data/segal_species/diet_mb_04_visit.pkl


In [75]:
print(diet_mb_02_visit.shape)
print(diet_mb_04_visit.shape)

(4482, 1441)
(1793, 1441)


In [ ]:
diet_mb.to_pickle(home_path + f"data/{SPECIES}/diet_mb.pkl")
with open(home_path + f'data/covariates.pkl', 'wb') as file:
    pickle.dump([bmi, comorbidities, lifestyle], file)

# Save my_lists.pkl with base_features+all covariates and all_features+all covariates
# Assumes base_features and all_features exist, and "covariates" means bmi+comorbidities+lifestyle columns as vector
all_covariate_cols = []
all_covariate_cols.append(bmi)
all_covariate_cols.extend(list(comorbidities))
# all_covariate_cols.extend(list(lifestyle))

base_features_with_covs = list(base_features) + all_covariate_cols
all_features_with_covs = list(all_features) + all_covariate_cols
print(base_features_with_covs)
with open(home_path + f'data/{SPECIES}/my_lists.pkl', 'wb') as file:
    pickle.dump([base_features_with_covs, all_features_with_covs, targets], file)

['age', 'sex', 'bmi', 'Obesity', 'Prediabetes', 'Hyperlipidemia', 'Hypertension', 'Irritable Bowel Syndrome (IBS)', 'Peptic Ulcer Disease', 'Gallstone disease', 'Asthma', 'Atopic dermatitis', 'Psoriasis', 'Allergy', 'Depression', 'Anxiety', 'Hypothyroidism']


In [78]:
# Save all changes dataframes
# Create baseline changes (all zeros, same structure as other changes)
# Demographics baseline changes
if 'demog_baseline' in globals() and 'demog_02_visit_change' in globals():
    demog_baseline_change = pd.DataFrame(
        0, 
        index=demog_baseline.index, 
        columns=demog_02_visit_change.columns
    )
    demog_baseline_change.to_pickle(home_path + f"data/demog_baseline_change.pkl")
    demog_02_visit_change.to_pickle(home_path + f"data/demog_02_visit_change.pkl")
    demog_04_visit_change.to_pickle(home_path + f"data/demog_04_visit_change.pkl")
    demog_change_cols = list(demog_02_visit_change.columns)
    print("Saved demographics changes")

# Lifestyle baseline changes
if 'lifestyle_baseline' in globals() and 'lifestyle_02_visit_change' in globals():
    lifestyle_baseline_change = pd.DataFrame(
        0, 
        index=lifestyle_baseline.index, 
        columns=lifestyle_02_visit_change.columns
    )
    lifestyle_baseline_change.to_pickle(home_path + f"data/lifestyle_baseline_change.pkl")
    lifestyle_02_visit_change.to_pickle(home_path + f"data/lifestyle_02_visit_change.pkl")
    lifestyle_04_visit_change.to_pickle(home_path + f"data/lifestyle_04_visit_change.pkl")
    lifestyle_change_cols = list(lifestyle_02_visit_change.columns)
    print("Saved lifestyle changes")

# Medications baseline changes
if 'mb_sub' in globals() and 'medications_02_visit_change' in globals():
    medications_baseline_change = pd.DataFrame(
        0, 
        index=mb_sub.index, 
        columns=medications_02_visit_change.columns
    )
    medications_baseline_change.to_pickle(home_path + f"data/medications_baseline_change.pkl")
    medications_02_visit_change.to_pickle(home_path + f"data/medications_02_visit_change.pkl")
    medications_04_visit_change.to_pickle(home_path + f"data/medications_04_visit_change.pkl")
    medications_change_cols = list(medications_02_visit_change.columns)
    print("Saved medications changes")

# Save column lists for change dataframes
if 'demog_change_cols' in globals() and 'lifestyle_change_cols' in globals() and 'medications_change_cols' in globals():
    with open(home_path + 'data/change_covariates.pkl', 'wb') as file:
        pickle.dump([demog_change_cols, lifestyle_change_cols, medications_change_cols], file)
    print("Saved change covariates column lists")


Saved demographics changes
Saved lifestyle changes
Saved medications changes
Saved change covariates column lists
